# Feature baseline (roadmap step 4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/feature_baseline.ipynb)

Trains the feature model ("Model B") on the corpus in your Drive and scores it on the same
held-out splits as the step 3 baselines:

* `features-nolm`: about 40 style features + Cosine Delta into LightGBM (CPU only, quick)
* `features`: the same plus GPT-2 perplexity, burstiness and token-rank features (needs a GPU)

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU. Then **Runtime → Run all**.

The GPT-2 pass over the whole corpus is the slow part (roughly 30 to 60 minutes on a T4). Its
results are saved to `My Drive/skolfie/cache/` every 20,000 texts, so if Colab disconnects,
just run all again and it continues where it stopped.

Reports go to `My Drive/skolfie/reports/<model>/` (with predictions for step 6), trained models
to `My Drive/skolfie/models/`.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
CACHE_DIR = "/content/drive/MyDrive/skolfie/cache"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

# features-nolm runs first (CPU, quick); features adds GPT-2 (GPU).
# Add "features-rf" to compare a random forest on the same features.
MODELS = ["features-nolm", "features"]
LM_BATCH_SIZE = 32  # lower to 16 if you get a CUDA out-of-memory error
MAX_TRAIN = None    # e.g. 50000 for a quick trial run; None = full train split
MAX_EVAL = None     # e.g. 20000 to cap each test split; None = everything

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. 'features-nolm' still works, but 'features' (GPT-2) would take "
          "many hours on CPU. Switch to a T4 GPU runtime, or remove 'features' from MODELS.")

from google.colab import drive

drive.mount("/content/drive")
for d in (REPORTS_DIR, CACHE_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."
assert os.path.exists(f"{DATA_DIR}/splits.parquet"), "Run evaluate.ipynb (step 3) first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Train and score

Each model prints its progress and report card. GPT-2 is downloaded from Hugging Face the
first time (about 500 MB); set the `HF_TOKEN` Colab secret if downloads are rate-limited.

In [ ]:
import sys

hf_token = secret("HF_TOKEN")
env = dict(os.environ, **({"HF_TOKEN": hf_token} if hf_token else {}))


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


extra = []
if MAX_TRAIN:
    extra += ["--max-train", str(MAX_TRAIN)]
if MAX_EVAL:
    extra += ["--max-eval", str(MAX_EVAL)]

for model in MODELS:
    print(f"\n========== {model} ==========", flush=True)
    run(["python", "-u", "scripts/evaluate.py",
         "--corpus", f"{DATA_DIR}/corpus.parquet",
         "--splits", f"{DATA_DIR}/splits.parquet",
         "--model", model,
         "--out-dir", f"{REPORTS_DIR}/{model}",
         "--cache-dir", CACHE_DIR,
         "--lm-batch-size", str(LM_BATCH_SIZE),
         "--save-model", f"{MODELS_DIR}/{model}.joblib",
         "--save-predictions", *extra])

## 4. Compare with the step 3 baselines

In [ ]:
import glob

reports = sorted(glob.glob(f"{REPORTS_DIR}/*/report.json"))
out = subprocess.run(["python", "scripts/evaluate.py", "--compare", *reports],
                     check=True, capture_output=True, text=True).stdout
print(out)
with open(f"{REPORTS_DIR}/comparison.md", "w") as f:
    f.write(out)

Reports are in `My Drive/skolfie/reports/`. Tell Claude in the project thread when this has
finished; it reads the report cards from Drive directly.